# API & CLI usage examples

This notebook is deliberately separate from the use-case notebooks (01–05): it's a reference for `agenttune`'s **public SDK surface** (`agenttune.api`) and **CLI** (`agenttune ...`), not a use case.

Covers:
1. `agenttune.api.run_pipeline` / `arun_pipeline` — run a DECIDE pipeline once
2. `PipelineResult` — the return type and its `.to_dict()`
3. `agenttune.api.train_agentic` — validation behavior (real error path)
4. The `agenttune` CLI: `version`, `pipeline`, `train` — via `typer.testing.CliRunner`, with a real model behind the `pipeline` subcommand


In [1]:
import sys, os, tempfile
sys.path.insert(0, ".")
import yaml
import unittest.mock as mock
from _real_backends import real_generate

import agenttune
from agenttune import api
print("agenttune version:", agenttune.__version__)
print("public api surface:", api.__all__ if hasattr(api, "__all__") else dir(api))


agenttune version: 0.1.0
public api surface: ['PipelineResult', 'run_pipeline', 'arun_pipeline', 'train_agentic']


## 1. `agenttune.api.arun_pipeline` (and its sync wrapper `run_pipeline`)

`run_pipeline`/`arun_pipeline(template, input_text, *, config=None) -> PipelineResult`. `template` can be a path to a YAML file or a registered template id (e.g. `"bfsi/kyc_triage"`, if present in your templates directory).

In [2]:
DECISION_SYSTEM = (
    'You are a JSON-only API for content moderation decisions. You ONLY ever output '
    'a single JSON object shaped exactly like {"decision": "approve"} or '
    '{"decision": "reject"}. Never output any other words, explanation, or punctuation.\n\n'
    'Example\nInput: "Hello, how is your day?"\nOutput: {"decision": "approve"}'
)

async def real_acompletion(*args, **kwargs):
    messages = kwargs.get("messages") or (args[0] if args else [])
    prompt = messages[-1]["content"]
    text = real_generate(prompt, system=DECISION_SYSTEM, max_new_tokens=20)
    return {"choices": [{"message": {"content": text}}]}

def write_simple_template(tmpdir):
    template = {
        "id": "docs/simple", "name": "Simple Example", "version": "1.0.0",
        "stages": [
            {"id": "s1", "type": "llm_call", "model": "gpt-4",
             "prompt": "Moderate: {input_text}", "max_iterations": 1},
            {"id": "s2", "type": "output", "verdict_field": "s1.output.decision", "destinations": []},
        ],
        "edges": [{"from": "s1", "to": "s2", "condition": None},
                  {"from": "s2", "to": "__end__", "condition": None}],
    }
    tdir = os.path.join(tmpdir, "templates", "docs")
    os.makedirs(tdir, exist_ok=True)
    tpath = os.path.join(tdir, "simple.yaml")
    with open(tpath, "w") as f:
        yaml.dump(template, f)
    cpath = os.path.join(tmpdir, "config.yaml")
    with open(cpath, "w") as f:
        yaml.dump({"api_keys": {"openai": "unused"}}, f)
    return tpath, cpath

tmp = tempfile.mkdtemp()
tpath, cpath = write_simple_template(tmp)

with mock.patch("agenttune.decide.stages.base.litellm.acompletion", side_effect=real_acompletion):
    result = await api.arun_pipeline(tpath, "A friendly customer service greeting.", config=cpath)

print(result)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PipelineResult(pipeline_id='32a571b7-c358-4c31-88af-1d801e188c2a', template_id='docs/simple', verdict='approve', verdict_label=None, confidence=None, reason=None, step_count=2, elapsed_seconds=40.470133, is_complete=True, error=None)


## 2. `PipelineResult` — the return type

In [3]:
print("verdict:       ", result.verdict)
print("verdict_label: ", result.verdict_label)
print("confidence:    ", result.confidence)
print("step_count:    ", result.step_count)
print("elapsed_sec:   ", result.elapsed_seconds)
print("is_complete:   ", result.is_complete)
print("error:         ", result.error)
print()
print("to_dict():", result.to_dict())


verdict:        approve
verdict_label:  None
confidence:     None
step_count:     2
elapsed_sec:    40.470133
is_complete:    True
error:          None

to_dict(): {'pipeline_id': '32a571b7-c358-4c31-88af-1d801e188c2a', 'template_id': 'docs/simple', 'verdict': 'approve', 'verdict_label': None, 'confidence': None, 'reason': None, 'step_count': 2, 'elapsed_seconds': 40.470133, 'is_complete': True, 'error': None}


## 3. `agenttune.api.train_agentic` — a real validation error path

`train_agentic(algorithm, **kwargs)` builds a trainer for `algorithm in {grpo, dpo, ppo, rloo, bco}`. Passing an unknown algorithm is a real, immediate `ValueError` naming the valid choices — no GPU/model needed to see this.

In [4]:
try:
    api.train_agentic("not_a_real_algorithm", model="x")
except ValueError as e:
    print(f"real ValueError: {e}")


real ValueError: Unknown algorithm 'not_a_real_algorithm'. Valid algorithms: grpo, dpo, ppo, rloo, bco.


## 4. The CLI — `agenttune version` / `pipeline` / `train`

Driven via `typer.testing.CliRunner`, exactly like a real terminal invocation of the installed `agenttune` command.

**Notebook-only wrinkle:** the `pipeline` subcommand calls the *synchronous* `run_pipeline`, which internally does `asyncio.run(...)`. That call fails inside **any** Jupyter kernel, because ipykernel always has an event loop running (`RuntimeError: asyncio.run() cannot be called from a running event loop`) — this is not specific to this notebook having used `await` earlier. A real terminal has no such loop, so `agenttune pipeline ...` works there with zero workaround; here we run the `CliRunner` call in a plain background thread (no event loop) to sidestep it.

In [5]:
import concurrent.futures

from typer.testing import CliRunner
from agenttune.cli.unified import app

runner = CliRunner()

result = runner.invoke(app, ["version"])
print("`agenttune version` ->", result.stdout.strip(), " exit_code:", result.exit_code)


`agenttune version` -> 0.1.0  exit_code: 0


In [6]:
out_path = os.path.join(tmp, "decision.json")

def _invoke_pipeline_cli():
    with mock.patch("agenttune.decide.stages.base.litellm.acompletion", side_effect=real_acompletion):
        return runner.invoke(app, [
            "pipeline", "--template", tpath, "--input", "This is spam, buy now!!!",
            "--config", cpath, "--output", out_path,
        ])

with concurrent.futures.ThreadPoolExecutor(1) as ex:
    result = ex.submit(_invoke_pipeline_cli).result()

print(f"`agenttune pipeline ...` exit_code={result.exit_code}")
print(result.stdout)
print("output file written:", os.path.exists(out_path))
if os.path.exists(out_path):
    with open(out_path) as f:
        print(f.read())


`agenttune pipeline ...` exit_code=0
Decision: None
Output:   /tmp/tmp3_vfm3h6/decision.json

output file written: True
{
  "pipeline_id": "e256552d-1dc7-45a4-bf22-c970bf78db99",
  "template_id": "docs/simple",
  "verdict": null,
  "verdict_label": null,
  "confidence": null,
  "reason": null,
  "step_count": 2,
  "elapsed_seconds": 0.042598,
  "is_complete": true,
  "error": null
}


In [7]:
result = runner.invoke(app, ["train", "--algorithm", "bogus", "--model", "x", "--dataset", "y"])
print(f"`agenttune train --algorithm bogus ...` exit_code={result.exit_code}")
print(result.stdout)


`agenttune train --algorithm bogus ...` exit_code=2
Unknown algorithm 'bogus'. Valid algorithms: grpo, dpo, ppo, rloo, bco.



---
**Notes for real production use (not simulated here):**
- `train_agentic`/`agenttune train` with a *real* algorithm (`grpo`, `dpo`, `ppo`, `rloo`, `bco`) needs `trl`/compatible `torch`/GPU installed — needs the exact version pins (`torch==2.11.0`, `trl[vllm]==1.7.1`, `vllm==0.23.0`) verified compatible on this box.
- `run_pipeline`/`arun_pipeline` against a real hosted model (OpenAI/Anthropic/etc via litellm) needs a real `api_keys` entry in `config.yaml` instead of the local-model patch used above — the patch exists here purely so this notebook runs without network access or an API key.
- There is currently **no CLI/`api.py` entry point for the self-heal closed loop** (`SelfHealLoop` / `decide.closed_loop.*`) — see notebook 03 for how to drive it directly.
